# Only once per runtime

## Installs

In [1]:
# !pip install -q -U transformers
# !pip install -q sentence-transformers
!pip install -q "transformers==4.46.3" sentence-transformers duckduckgo-search tqdm lancedb gptqmodel "autoawq<0.2.9" "protobuf<6.0" "numpy<2.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 32.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.6/885.6 kB 58.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.8/882.8 kB 44.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 685.5/685.5 kB 53.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to bui

## Load dataset on content from drive

In [18]:
!apt-get install -y pv

# 1. Create target directories safely
!mkdir -p /content/db_local /content/db_local_extracted

# 2. Copy from Drive with a native progress bar (takes around 7 minutes)
print("--> Copying zip file from Google Drive...\n Estimated duration: 7:30 min")
!rsync -ah --progress /content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local.zip /content/db_local/

# 3. Unzip with a progress bar based on data streaming (Low Overhead)
print("\n--> Extracting archive contents...\n Estimated duration: 8 min")
!7z x /content/db_local/db_local.zip -o/content/db_local_extracted/ -bsp1

VECTOR_DB_PATH = "/content/db_local_extracted/db_local"

^C
Archive:  /content/db_local
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/db_local or
        /content/db_local.zip, and cannot find /content/db_local.ZIP, period.


# Every session

## Connection to PoliMilionaire

### Imports

In [3]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata, drive
from huggingface_hub import login
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Callable
import os
import pandas as pd
from transformers import AutoModelForSeq2SeqLM,AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig
from sentence_transformers import util
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
import tqdm
from datasets import load_dataset, load_from_disk
import lancedb
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS

### Connections

#### Google Drive

In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


#### Hugging Face

In [7]:
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


#### Game APIs

In [8]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 227 (delta 33), reused 11 (delta 2), pack-reused 168 (from 1)
Receiving objects: 100% (227/227), 818.45 KiB | 4.96 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi
Logged in as: GliEmbeddingRuspanti (role: student)


#### Load dataset on content from drive

In [9]:
# !cp -r /content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local.zip /content/db_local
# !unzip /content/db_local -d /content/db_local_extracted
# VECTOR_DB_PATH = "/content/db_local_extracted/db_local"


### Model class

In [10]:
class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"


### The Game

In [11]:
def play_game(game, model, sys_prompt):
  log = []
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      options = {f"{opt.id}": opt.text for opt in question.options}

      t0 = time.time()
      answer_summary, answer_input = model.answer(question.text, options, sys_prompt)
      inference_time = time.time() - t0
      print(f"Model answer: {answer_input}")
      answer_id = int(answer_input)

      choosen_answer = question.options[answer_id]

      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

      # Log the outcome
      entry = {
          'level'           : game.current_level,
          'question'        : question.text,
          'options'         : question.options,
          'chosen_option'   : choosen_answer.text,
          'correct'         : result.correct,
          'timed_out'       : result.timed_out,
          'inference_time'  : round(inference_time, 2),
          'answer_summary'  : answer_summary,
      }
      log.append(entry)

  summary = {
        'model'           : model.name,
        'final_level'     : game.current_level,
        'earned_amount'   : game.earned_amount,
        'num_questions'   : len(log),
        'num_correct'     : sum(1 for e in log if e['correct']),
        'num_timed_out'   : sum(1 for e in log if e['timed_out']),
        'avg_inference_s' : round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'             : log,
    }

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

  return summary

### RAG model class

#### Web Retireval

In [12]:
def _scrape_page(url: str, max_chars: int = 3000) -> str:
    try:
        r = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)[:max_chars]
    except Exception:
        return ""

def _fetch_web_documents(query: str, k: int = 10, full_text: bool = True) -> list[str]:
    """Cerca su DuckDuckGo e restituisce una lista di testi (snippet o pagine complete)."""
    docs = []
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=k))

    for r in results:
        if full_text:
            text = _scrape_page(r["href"])
            docs.append(text if text else r["body"])  # fallback sullo snippet
        else:
            docs.append(r["body"])  # solo snippet (~200 char)

    return docs

#### Load model

In [13]:
def load_rag_model(use_db: bool = True):
  bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16})
  reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=1024)
  model_id = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
  )
  collection = ds = None
  if use_db:
    if not os.path.exists(VECTOR_DB_PATH):
      VECTOR_DB_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local/"
    table_name = "wiki_rag_collection"
    if not os.path.exists(VECTOR_DB_PATH):
      print("ERRORE CRITICO: La cartella base non esiste per Colab!")
    else:
        contenuto = os.listdir(VECTOR_DB_PATH)
        print(f"Cosa c'è fisicamente dentro '{VECTOR_DB_PATH}':")
        print(contenuto)

        if f"{table_name}.lance" in contenuto:
            print(f"\n✅ PERFETTO! La tabella fisica '{table_name}.lance' C'È.")
        else:
            print(f"\n❌ ERRORE! La tabella fisica '{table_name}.lance' MANCA in questa cartella.")
            print("Probabilmente il path è sbagliato o la cartella è nidificata più a fondo.")

    print("\n--- TEST LANCEDB ---")
    db = lancedb.connect(VECTOR_DB_PATH)

    # TRUCCO: Usa table_names() invece di list_tables()
    tabelle_presenti = db.table_names()
    print(f"Tabelle viste da LanceDB: {tabelle_presenti}")

    if table_name in tabelle_presenti:
        collection = db.open_table(table_name)
        print(f"🎉 SUCCESSO! Tabella '{table_name}' aperta.")
    else:
        print(f"❌ FALLIMENTO. LanceDB non vede la tabella.")

    # Assuming DS_PATH (directory for Arrow cache) and PARQUET_PATH are defined earlier
    DS_PATH="/content/drive/MyDrive/Progetto-NLP/Branch-rag/"
    PARQUET_PATH = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'

    # We save Hugging Face datasets as a directory structure, not a single file
    ds_arrow_dir = os.path.join(DS_PATH, "ds_embedding_collection_ita")

    # ds = None

    # Attempt to load from native Hugging Face Disk Cache (Super Fast Arrow Format)
    if os.path.exists(ds_arrow_dir):
        print("Attempting to load dataset from native disk cache...")
        try:
            ds = load_from_disk(ds_arrow_dir)
            _ = len(ds)  # Quick verification
            print("Dataset loaded successfully from disk cache.")
        except Exception as e:
            print(f"Failed to load dataset from cache ({e}). Attempting to load from raw Parquet instead.")
            ds = None

    # Fallback: If cache doesn't exist or is corrupted, load from Parquet
    if ds is None:
        if os.path.exists(PARQUET_PATH):
            print("Loading dataset from Parquet...")
            ds = load_dataset("parquet", data_files=PARQUET_PATH, split="train")
            print("Dataset loaded successfully from Parquet.")

            # Save it natively to disk for blazing fast future loading
            print("Caching dataset to disk for future use...")
            ds.save_to_disk(ds_arrow_dir)
            print("Dataset cached successfully.")
        else:
            print(f"Error: Neither cache directory ({ds_arrow_dir}) nor Parquet file ({PARQUET_PATH}) found.")
            raise FileNotFoundError(f"Cannot load dataset. Parquet file not found at {PARQUET_PATH}")

  return bi_enc, reranker, model, tokenizer, collection, ds



#### Query processing

In [ ]:
def translate_and_rewrite_query(self, query):
    # 1. Prompt "Aggressivo" per Traduzione e Ottimizzazione
    system_prompt = (
        "Sei un traduttore esperto e un ottimizzatore per motori di ricerca. "
        "Il tuo compito è tradurre la domanda dell'utente in ITALIANO e riscriverla "
        "in modo chiaro e descrittivo, ideale per cercare in un database di Wikipedia in italiano. "
        "REGOLE FONDAMENTALI:\n"
        "1. L'output deve essere ESCLUSIVAMENTE in lingua italiana.\n"
        "2. Restituisci SOLO la domanda tradotta e ottimizzata.\n"
        "3. Non aggiungere saluti, spiegazioni, 'Ecco la traduzione' o virgolette."
    )

    user_prompt = f"Traduci e ottimizza la seguente domanda:\n<domanda>\n{query}\n</domanda>\n\nDomanda in italiano:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Trasformazione in tensori
    inputs = self.tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(self.model.device)

    # 2. Generazione (Max 60 token, niente creatività)
    outputs = self.model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        use_cache=True,
        pad_token_id=self.tokenizer.eos_token_id
    )

    torch.cuda.synchronize()

    # Decodifica
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    query_ita = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Pulizia extra
    query_ita = query_ita.replace('"', '').replace("'", "")

    return query_ita

#### Query answering logic

In [14]:
def rag_sota(self, query, options_text, top_k=2):
    start_time = time.time()
    print("\nInizio retrieval...")

    # A1. RETRIEVAL (LanceDB)
    if self.use_db:
      processed_query = translate_and_rewrite_query(self, query)
      query_vector = self.bi_enc.encode([processed_query]).tolist()[0]
      risultati = self.collection.search(query_vector).limit(20).to_pandas()
      retrieved_docs = []
      # B. PREPARAZIONE DOCUMENTI
      for _, row in risultati.iterrows():
          doc_id = int(row['id'])
          retrieved_docs.append(self.ds[doc_id]['content'])
          retrieved_docs = [doc for doc in retrieved_docs if len(doc.strip()) > 50]

    # A2. RETRIEVAL (Web)
    if self.use_web:
      retrieved_docs = _fetch_web_documents(query, k=self.top_k_web, full_text=self.full_text)
      print(f"Recuperati {len(retrieved_docs)} documenti dal web.")


    # C. RERANKING
    couples = [[query, doc] for doc in retrieved_docs]
    scores = self.reranker.predict(couples)

    docs_with_score = list(zip(scores, retrieved_docs))
    docs_with_score.sort(key=lambda x: x[0], reverse=True)

    # D. TOP K DOCS (con TRUNCATION DI SICUREZZA)
    top_docs = [doc for score, doc in docs_with_score[:top_k]]
    docs_context = "\n\n---\n\n".join(top_docs)

    if len(docs_context) > 12000:
        docs_context = docs_context[:12000] + "\n... [TRONCATO PER SICUREZZA]"
        print("TRONCATO")

    print("Retrieval finito.")
    end_time_retrivial = time.time()

    # E. PULIZIA RAM
    try:
      del risultati
    except NameError:
      pass
    del retrieved_docs
    del couples
    #gc.collect()
    torch.cuda.empty_cache()

    # # F. PROMPTING
    # system_prompt = "Sei un risolutore di quiz. Leggi il contesto, ragionaci e restituisci ESCLUSIVAMENTE il numero dell'opzione corretta. Non scrivere altro."

    # user_prompt = f"""<contesto>
    #   {docs_context}
    #   </contesto>

    #   Domanda: {query}
    #   Opzioni: {options_text}

    #   Istruzione: Il contesto è in parte in italiano e in parte in inglese, le opzioni in inglese. Analizza il contesto, ragionaci e scrivi SOLO il numero dell'opzione corretta.
    #   Risposta:"""

    # if not self.use_db:
    system_prompt = "You are a quiz solver. Read the context and answer the question. Do not use external knowledge."

    user_prompt = f"""<context>
{docs_context}
</context>

Question: {query}
Options: {options_text}

Instructions: The context can be in Italian or in English, the options are in English. Analyze the context, reason on its meaning and write ONLY the ID of the correct option.
Answer:"""


    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_testo = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # MODIFICA 1: Chiamiamo la variabile 'inputs' e aggiungiamo return_dict=True
    inputs = self.tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True # <-- FONDAMENTALE
    ).to(self.model.device)

    # G. INFERENZA NATIVA ULTRA-VELOCE
    outputs = self.model.generate(
        **inputs,              # <-- MODIFICA 2: Spacchettiamo il dizionario con i due asterischi!
        max_new_tokens=10,
        do_sample=False,
        use_cache=True,
        pad_token_id=self.tokenizer.eos_token_id
    )

    # MODIFICA 3: Dobbiamo prendere la lunghezza da inputs['input_ids']
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    predicted_answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    end_time = time.time()
    tempo_esecuzione = end_time - start_time
    tempo_retrivial = end_time_retrivial - start_time
    print(f"Tempo retrivial: {tempo_retrivial:.2f} secondi")
    print(f"Tempo di esecuzione totale: {tempo_esecuzione:.2f} secondi")

    return prompt_testo, predicted_answer, top_docs

#### RAG model class definition

In [15]:
class RAGModel(Model):
  def __init__(self, name: str, use_db: bool = True, use_web: bool = True, top_k_web: int = 20, full_text: bool = True):
    self.name=name
    self.use_db=use_db
    self.use_web=use_web
    self.top_k_web=top_k_web
    self.full_text=full_text
    self.bi_enc, self.reranker, self.model, self.tokenizer, self.collection, self.ds = load_rag_model(use_db)

  def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
    _, answer, _ = rag_sota(self, question, options_text=options)
    return _, answer

## Instantiate model

In [17]:
rag_model = RAGModel("RAG")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

ERRORE CRITICO: La cartella base non esiste per Colab!

--- TEST LANCEDB ---
Tabelle viste da LanceDB: []
❌ FALLIMENTO. LanceDB non vede la tabella.


/tmp/ipykernel_1591/3322052528.py:32: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  tabelle_presenti = db.table_names()


Attempting to load dataset from native disk cache...
Dataset loaded successfully from disk cache.


In [ ]:
models = {
    "RAG": {"model": rag_model, "system_prompt": ""}}

## Play and print results

In [ ]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")

    model = config["model"]
    system_prompt = config["system_prompt"]

    model_results = []

    for comp_id in [0, 1, 2, 3]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id)

        summary = play_game(game, model, system_prompt)

        model_results.append(summary)

    results[model_name] = model_results

In [ ]:
def print_results(results):
    for model_name, competitions in results.items():

        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            confidence_array = []

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                # ─────────────────────────────────────────────
                # CONFIDENCE EXTRACTION (robust fallback chain)
                # ─────────────────────────────────────────────
                answer_summary = entry.get("answer_summary", {})

                if isinstance(answer_summary, dict):
                    if "normalized_margin" in answer_summary:
                        conf = answer_summary["normalized_margin"]

                    elif "confidence" in answer_summary:
                        conf = answer_summary["confidence"]

                    else:
                        conf = None
                else:
                    conf = None

                confidence_array.append(conf)

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s "
                    f"Conf: {conf if conf is not None else 'N/A'}"
                )

            # ─────────────────────────────────────────────
            # PRINT SUMMARY CONFIDENCE ARRAY
            # ─────────────────────────────────────────────
            print("\n📊 Confidence Array:")
            if any(c is not None for c in confidence_array):
                print(confidence_array)
            else:
                print("Confidence not available")

        print("\n")

# final print
print_results(results)